# SVS pipeline — controlled singing-voice stimuli for LMC

The text-to-song route (ACE-Step) failed the experiment's hard requirements: lyrics
weren't verbatim/intelligible, the voice drifted, and requested tempo/emotion were
ignored. **Singing-voice synthesis flips the problem** — we *specify* exact notes,
exact lyrics, and one fixed voice, so lyrics + voice are guaranteed by construction and
only the musical emotion is manipulated (melody varies per emotion, as chosen).

This notebook does everything up to the vocal render, headlessly:
**hook → syllables → per-emotion melody → Score → MusicXML**. You then render the
MusicXML in the SVS app with one fixed voice, and validate with the same WER / VA
harness built for the ACE-Step arm.

## 0 · Setup

In [8]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # `lmcgen`, `lmcsvs`
from lmcsvs import config as C, syllables, melody, score, musicxml, pipeline, validate
import logging; logging.basicConfig(level=logging.INFO)
C.ensure_dirs()
print(C.summary())

Repo root   : /Users/budge.13/Desktop/Music Analysis
SVS dir     : /Users/budge.13/Desktop/Music Analysis/data/svs
Scores      : /Users/budge.13/Desktop/Music Analysis/data/svs/scores  (MusicXML for the SVS app)
Emotions    : ecstasy, admiration, terror, amazement, grief, loathing, rage, vigilance
Grid        : 8 lyrics x 8 melodies = 64 scores
Engine      : synthv  (voice: (assign one fixed voice in the app))
Melody      : 2 bars, base octave 4, one syllable/note
Also MIDI   : False


> **Recommended:** `pip install pyphen` for accurate syllabification (a naive fallback
> runs without it). To subset to a 2×2 valence×arousal design, set e.g.
> `LMCSVS_EMOTIONS="rage,ecstasy,grief,admiration"` before importing.

## 1 · Inspect a cell (hook → syllables → melody → MusicXML)

In [9]:
sc = score.assemble("grief", "grief")          # congruent cell
print("syllables:", [n.lyric for n in sc.notes])
print("MIDI     :", [n.midi for n in sc.notes])
print(f"tempo={sc.tempo_bpm}  key={sc.keyscale}  notes={len(sc.notes)}")
print(musicxml.to_musicxml(sc)[:600], "…")

syllables: ['The', 'house', 'is', 'qui', 'et', 'where', 'you', 'used', 'to', 'be', 'I', 'keep', 'your', 'ghost', 'for', 'com', 'pa', 'ny']
MIDI     : [58, 60, 58, 56, 55, 56, 58, 56, 55, 55, 56, 55, 56, 58, 60, 62, 60, 60]
tempo=68  key=C minor  notes=18
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 3.1 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">
<score-partwise version="3.1"><work><work-title>grief lyric / grief music</work-title></work><part-list><score-part id="P1"><part-name>Voice</part-name></score-part></part-list><part id="P1"><measure number="1"><attributes><divisions>4</divisions><key><fifths>-3</fifths><mode>minor</mode></key><time><beats>4</beats><beat-type>4</beat-type></time><clef><sign>G</sign><line>2</line></clef></attributes><direction placement="above"><direct …


## 2 · Export all scores for the SVS app

In [10]:
paths = pipeline.export_scores()
print(f"Exported {len(paths)} MusicXML scores to {C.SCORE_DIR}")

INFO:lmcsvs.pipeline:Exported 64 MusicXML scores → /Users/budge.13/Desktop/Music Analysis/data/svs/scores
INFO:lmcsvs.pipeline:Next: import into synthv, assign one fixed voice, batch-render to /Users/budge.13/Desktop/Music Analysis/data/svs/audio


Exported 64 MusicXML scores to /Users/budge.13/Desktop/Music Analysis/data/svs/scores


## 3 · Render in the SVS app  *(manual, one time)*

1. Open **Synthesizer V Studio 2 Pro** (or ACE Studio).
2. **Import** each `data/svs/scores/<L>__<M>.musicxml` — notes arrive with the lyrics
   already attached to each note.
3. Assign **one fixed voice** to every score (never change it — this is the voice
   control). Record it in `LMCSVS_VOICE`.
4. Optionally add per-emotion expression, but keep the voice/settings identical across
   cells except what the score dictates.
5. **Batch-render** to WAV → save as `data/svs/audio/<L>__<M>.wav`.

Batch tip: SynthV Studio 2 Pro supports JavaScript/Lua scripts; a render-all script can
automate step 5 across imported projects.

*(Backing instrumental — the accompaniment that carries musical emotion alongside the
melody — is a separate stage; the rendered vocal alone validates lyric intelligibility
and the melody's valence/arousal first.)*

## 3b · Headless render with DiffSinger  *(fully scriptable alternative to SynthV)*

No GUI: our scores render to vocals from Python. **One-time setup** (in a separate env):

1. Clone the **OpenVPI DiffSinger** repo and install its requirements in its own env.
2. Download an **English acoustic voicebank** (e.g. *Peiton*, ARPABET) into its `checkpoints/`.
3. Here: `pip install g2p_en` (ARPABET phonemes; a crude fallback runs without it).
4. Set env vars: `DIFFSINGER_REPO`, `DIFFSINGER_EXP` (experiment/voicebank name),
   `DIFFSINGER_SPK` (if multi-speaker), `DIFFSINGER_PYTHON` (its interpreter).

> **Phoneme caveat:** English DiffSinger banks use a bank-specific ARPABET+ dictionary.
> Verify one `.ds` against your bank first — the reliable check is to import our MusicXML
> into **OpenUtau** (free; it has the matching DIFFS-EN phonemizer and can render DiffSinger
> directly), which also works as a full fallback if the headless phoneme set mismatches.

In [11]:
from lmcsvs import diffsinger
# writes data/svs/ds/<L>__<M>.ds (inspect one before a full run)
ds_paths = diffsinger.export_ds()
print(len(ds_paths), '.ds files written to', C.DS_DIR)
# once DIFFSINGER_* env vars are set + the voicebank is installed:
# wavs = diffsinger.render()      # -> data/svs/audio/<L>__<M>.wav
# df   = validate.validate()      # then the same WER / VA harness

INFO:lmcsvs.diffsinger:Exported 64 .ds scores → /Users/budge.13/Desktop/Music Analysis/data/svs/ds


64 .ds files written to /Users/budge.13/Desktop/Music Analysis/data/svs/ds


## 3c · Render with OpenUtau + Peiton  *(the working English DiffSinger path)*

Headless English DiffSinger isn't really supported (the OpenVPI repo is PyTorch/Chinese; English voicebanks ship as ONNX for OpenUtau). OpenUtau bundles its own DiffSinger runtime, so this is the reliable route — and it reuses the MusicXML we already export.

**One-time setup**
1. Install **OpenUtau** — open `~/Desktop/OpenUtau-osx-arm64.dmg`, drag OpenUtau to Applications. (Unsigned app: right-click → Open the first time, or run `xattr -dr com.apple.quarantine /Applications/OpenUtau.app`.)
2. Get **Peiton DS** — read the Terms of Use, then download from the creator's page and install via OpenUtau **Menu → Tools → Install Singer** (or **Install Dependency** for the .oudiffsinger). ⚠️ Peiton is *free for use* but *commercial use needs a purchased license* — confirm the ToU covers your study + any stimulus sharing before you rely on it.
3. In OpenUtau set the phonemizer to **DIFFS EN (ARPA)** so lyrics map to Peiton's phonemes.

**Per render (one-time batch)**
4. `pipeline.export_scores()` (cell above) → `data/svs/scores/<L>__<M>.musicxml`.
5. In OpenUtau: **File → Import** a score → assign **Peiton** as the singer → the lyrics arrive attached to the notes.
6. **File → Export Mixdown (WAV)** → save as `data/svs/audio/<L>__<M>.wav` (match the names).
7. Back here: `validate.validate()` scores WER (should be ~0) + valence/arousal.

> **Start small:** render 2-3 cells first and listen — confirm Peiton's voice + intelligibility before doing the whole grid. If quality is good, then batch the rest.

> Note: the `~/Desktop/DiffSinger` PyTorch repo + `diffsinger` conda env from step 1 are **not used** by this OpenUtau route (OpenUtau has its own DiffSinger runtime) — you can delete them, or keep them for reference / a future headless PyTorch model.

## 4 · Validate rendered vocals (WER + valence/arousal)

In [12]:
# Run after you've rendered wavs into data/svs/audio/. Needs faster-whisper + librosa.
df = validate.validate()
if df.empty:
    print("No rendered wavs yet — render the scores first (step 3).")
else:
    print("WER median (should be ~0 — lyrics are sung by construction):",
          round(df["wer"].median(), 3))
    print("vocal present:", round(df["vocal_present"].mean(), 3))
    display(df[["lyric_emotion","music_emotion","congruent","wer","audio_v","audio_a"]].round(3).head(12))

No rendered wavs yet — render the scores first (step 3).


## 5 · Next steps

- **Backing track** per music-emotion (chords + drums in the emotion's key/tempo),
  mixed under the vocal, so the *music* carries emotion alongside the melody.
- **Human validation** of valence/arousal on the rendered clips (the ground-truth
  manipulation check).
- Feed the validated stimuli into the respondent study (liking / repeat-listening vs LMC).